In [ ]:
#Uncontrolled Scenario
import pandas as pd
import matplotlib.pyplot as plt
from pyswmm import Simulation
from pyswmm import Simulation, Nodes
from pyswmm import Simulation, Links
from pyswmm import Simulation, Subcatchments

file_path = r'E:\MsE U-M\Winter 24\Sensors\Project\test.inp'

timesteps = list()
outfall_data = list()
sub_data = list()
basins_pipe_data = list()
wbasin_data = list()
ebasin_data = list()

with Simulation(file_path) as sim:

    node_list = Nodes(sim)
    link_list = Links(sim)
    sub_list = Subcatchments(sim)

    sub = sub_list['Se']
    outfall = node_list['outlet']
    basins_pipe = link_list['C5']
    wbasin = node_list['west_basin']
    ebasin = node_list['east_basin']

    sim.step_advance(300)

    for step in sim:
        timesteps.append(sim.current_time)
        outfall_data.append(outfall.depth)
        basins_pipe_data.append(basins_pipe.depth)
        wbasin_data.append(wbasin.depth)
        ebasin_data.append(ebasin.depth)
        sub_data.append(sub.runoff)

ts1 = pd.Series(outfall_data, index=timesteps, name='Uncontrolled Outfall depth')
ts2 = pd.Series(basins_pipe_data, index=timesteps, name='Uncontrolled Basins Pipe depth')
ts3 = pd.Series(sub_data, index=timesteps, name='Subcatchment')
ts4 = pd.Series(wbasin_data, index=timesteps, name='Uncontrolled West Basin depth')
ts5 = pd.Series(ebasin_data, index=timesteps, name='Uncontrolled East Basin depth')

ts1.plot()
ts2.plot()
ts4.plot()
ts5.plot()
plt.legend()
plt.show()

In [ ]:
#Controlled Scenario
from pyswmm import Simulation, Nodes, Links
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from scipy.signal import find_peaks

file_path = r'E:\MsE U-M\Winter 24\Sensors\Project\test.inp'


class PIDController:
    def __init__(self, Kp, Ki, Kd, setpoint):
        self.Kp = Kp
        self.Ki = Ki
        self.Kd = Kd
        self.setpoint = setpoint
        self.prev_error = 0
        self.integral = 0

    def update(self, current_value):
        error = self.setpoint - current_value
        self.integral += error
        derivative = error - self.prev_error
        output = self.Kp * error + self.Ki * self.integral + self.Kd * derivative
        self.prev_error = error
        return output


# Define control parameters for outfall
threshold_outfall_depth = 0.6   # Example threshold for outfall depth
Kp_outfall = 1.8
Ki_outfall = 0.0
Kd_outfall = 0.0

# Define control parameters for basins pipe
threshold_basins_pipe_depth = 1.5   # Example threshold for basins pipe depth
Kp_basins_pipe = 0.8
Ki_basins_pipe = 0.0
Kd_basins_pipe = 0.0

# Example maximum and minimum valve settings (fully open and fully closed)
max_valve_setting = 1.0
min_valve_setting = 0.0


# Initialize lists for storing simulation data
timesteps = []
outfall_data = []
basins_pipe_data = []
wbasin_data = []
ebasin_data = []
outfall_valve_settings = []
basins_pipe_valve_settings = []

# Initialize PID controllers for outfall and basins pipe
outfall_pid = PIDController(Kp_outfall, Ki_outfall, Kd_outfall, threshold_outfall_depth)
basins_pipe_pid = PIDController(Kp_basins_pipe, Ki_basins_pipe, Kd_basins_pipe, threshold_basins_pipe_depth)


def control_logic(sim, basins_pipe, outfall, west_basin, east_basin):
    for step in sim:
        # Get current conditions
        current_time = sim.current_time
        outfall_depth = outfall.depth
        basins_pipe_depth = basins_pipe.depth
        west_basin_depth = west_basin.depth
        east_basin_depth = east_basin.depth

        # Update PID controllers for outfall and basins pipe
        outfall_setting = outfall_pid.update(outfall_depth)
        basins_pipe_setting = basins_pipe_pid.update(basins_pipe_depth)

        # Apply valve settings for outfall and basins pipe
        outfall.target_setting = max(min(outfall_setting, max_valve_setting), min_valve_setting)
        basins_pipe.target_setting = max(min(basins_pipe_setting, max_valve_setting), min_valve_setting)

        # Record valve settings
        outfall_valve_settings.append(outfall.target_setting)
        basins_pipe_valve_settings.append(basins_pipe.target_setting)

        # Append data for plotting
        timesteps.append(current_time)
        outfall_data.append(outfall_depth)
        basins_pipe_data.append(basins_pipe_depth)
        wbasin_data.append(west_basin.depth)
        ebasin_data.append(east_basin.depth)


# Run the simulation
with Simulation(file_path) as sim:
    node_list = Nodes(sim)
    link_list = Links(sim)

    outfall = node_list['outlet']
    basins_pipe = link_list['C5']
    west_basin = node_list['west_basin']
    east_basin = node_list['east_basin']

    sim.step_advance(80)

    # Implement control logic during simulation
    control_logic(sim, basins_pipe, outfall, west_basin, east_basin)

ts1 = pd.Series(outfall_data, index=timesteps, name='Outfall')
ts2 = pd.Series(basins_pipe_data, index=timesteps, name='Basins Pipe')
ts4 = pd.Series(wbasin_data, index=timesteps, name='West Basin')
ts5 = pd.Series(ebasin_data, index=timesteps, name='East Basin')

ts1.plot()
ts2.plot()
ts4.plot()
ts5.plot()
plt.legend()
plt.show()

# control signals for the valves
plt.plot(timesteps, outfall_valve_settings, label='Outfall Valve Setting')
plt.plot(timesteps, basins_pipe_valve_settings, label='Basins Pipe Valve Setting')
plt.xlabel('Time')
plt.ylabel('Valve Setting')
plt.title('Control Signals for Valves')
plt.legend()
plt.grid(True)
plt.show()